In [0]:
from pyspark.sql.functions import col, min, datediff,count,avg

maint = spark.table("workspace.default.silver_maint")
failures = spark.table("workspace.default.silver_failures")
machines = spark.table("workspace.default.silver_machines")

maint.printSchema()
failures.printSchema()
machines.printSchema()

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- comp: string (nullable = true)

root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- failure: string (nullable = true)

root
 |-- machineID: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- age: integer (nullable = true)



In [0]:
maintenance_failure = (
    maint.alias("m")
    .join(
        failures.alias("f"),
        (col("m.machineID") == col("f.machineID")) &
        (col("m.comp") == col("f.failure")) &
        (col("f.datetime") > col("m.datetime")),
        "left"
    )
    .groupBy(
        col("m.datetime").alias("maintenance_datetime"),
        col("m.machineID").alias("machineID"),
        col("m.comp").alias("component")
    )
    .agg(
        min("f.datetime").alias("next_failure_datetime")
    )
)

print("Maintenance events matched:", maintenance_failure.count())

display(maintenance_failure.limit(10))

Maintenance events matched: 3286


maintenance_datetime,machineID,component,next_failure_datetime
2015-02-04T06:00:00.000Z,1,comp4,2015-06-19T06:00:00.000Z
2014-11-28T06:00:00.000Z,3,comp2,2015-01-07T06:00:00.000Z
2015-07-01T06:00:00.000Z,4,comp1,2015-08-30T06:00:00.000Z
2015-01-19T06:00:00.000Z,10,comp2,2015-04-04T06:00:00.000Z
2015-02-19T06:00:00.000Z,11,comp3,2015-04-20T06:00:00.000Z
2015-07-10T06:00:00.000Z,13,comp2,2015-08-09T06:00:00.000Z
2014-10-14T06:00:00.000Z,14,comp2,2015-01-31T06:00:00.000Z
2015-05-31T06:00:00.000Z,14,comp1,2015-07-15T06:00:00.000Z
2014-06-01T06:00:00.000Z,21,comp3,2015-01-23T06:00:00.000Z
2015-07-07T06:00:00.000Z,21,comp2,2015-08-06T06:00:00.000Z


In [0]:
mtbf_data = (
    maintenance_failure
    .withColumn(
        "mtbf_days",
        datediff(
            col("next_failure_datetime"),
            col("maintenance_datetime")
        )
    )
    .join(
        machines.select("machineID", "model", "age"),
        on="machineID",
        how="left"
    )
)

display(
    mtbf_data
    .select(
        "machineID",
        "model",
        "age",
        "component",
        "maintenance_datetime",
        "next_failure_datetime",
        "mtbf_days"
    )
    .limit(10)
)

machineID,model,age,component,maintenance_datetime,next_failure_datetime,mtbf_days
1,model3,18,comp4,2015-02-04T06:00:00.000Z,2015-06-19T06:00:00.000Z,135
3,model3,8,comp2,2014-11-28T06:00:00.000Z,2015-01-07T06:00:00.000Z,40
4,model3,7,comp1,2015-07-01T06:00:00.000Z,2015-08-30T06:00:00.000Z,60
10,model3,10,comp2,2015-01-19T06:00:00.000Z,2015-04-04T06:00:00.000Z,75
11,model2,6,comp3,2015-02-19T06:00:00.000Z,2015-04-20T06:00:00.000Z,60
13,model1,15,comp2,2015-07-10T06:00:00.000Z,2015-08-09T06:00:00.000Z,30
14,model3,1,comp2,2014-10-14T06:00:00.000Z,2015-01-31T06:00:00.000Z,109
14,model3,1,comp1,2015-05-31T06:00:00.000Z,2015-07-15T06:00:00.000Z,45
21,model2,14,comp3,2014-06-01T06:00:00.000Z,2015-01-23T06:00:00.000Z,236
21,model2,14,comp2,2015-07-07T06:00:00.000Z,2015-08-06T06:00:00.000Z,30


In [0]:
model_mtbf = (
    mtbf_data
    .filter(col("mtbf_days").isNotNull())
    .groupBy("model")
    .agg(
        count("*").alias("maintenance_events"),
        avg("mtbf_days").alias("average_mtbf_days")
    )
    .orderBy("model")
)

display(model_mtbf)

model,maintenance_events,average_mtbf_days
model1,314,103.05095541401273
model2,287,110.71777003484321
model3,454,116.90308370044053
model4,377,119.24933687002653


In [0]:
model_component_mtbf = (
    mtbf_data
    .filter(col("mtbf_days").isNotNull())
    .groupBy("model", "component")
    .agg(
        count("*").alias("maintenance_events"),
        avg("mtbf_days").alias("average_mtbf_days")
    )
    .orderBy("model", "component")
)

display(model_component_mtbf)

model,component,maintenance_events,average_mtbf_days
model1,comp1,59,123.13559322033899
model1,comp2,94,108.32978723404256
model1,comp3,94,88.92553191489361
model1,comp4,67,97.77611940298507
model2,comp1,58,107.25862068965517
model2,comp2,92,116.44565217391305
model2,comp3,89,109.82022471910112
model2,comp4,48,105.58333333333333
model3,comp1,143,139.04895104895104
model3,comp2,209,108.10047846889952


In [0]:
gold_q10 = (
    mtbf_data
    .select(
        "machineID",
        "model",
        "age",
        "component",
        "maintenance_datetime",
        "next_failure_datetime",
        "mtbf_days"
    )
)

gold_q10.printSchema()

root
 |-- machineID: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- maintenance_datetime: timestamp (nullable = true)
 |-- next_failure_datetime: timestamp (nullable = true)
 |-- mtbf_days: integer (nullable = true)



In [0]:
from delta.tables import DeltaTable

gold_table_name = "workspace.default.gold_q10_mtbf"

DeltaTable.createIfNotExists(spark) \
    .tableName(gold_table_name) \
    .addColumn("machineID", "INT") \
    .addColumn("model", "STRING") \
    .addColumn("age", "INT") \
    .addColumn("component", "STRING") \
    .addColumn("maintenance_datetime", "TIMESTAMP") \
    .addColumn("next_failure_datetime", "TIMESTAMP") \
    .addColumn("mtbf_days", "INT") \
    .execute()

print("Empty Gold Q10 Delta table created")

spark.table(gold_table_name).printSchema()

Empty Gold Q10 Delta table created
root
 |-- machineID: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- maintenance_datetime: timestamp (nullable = true)
 |-- next_failure_datetime: timestamp (nullable = true)
 |-- mtbf_days: integer (nullable = true)



In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.default.gold_q10_mtbf"
)

target.alias("t").merge(
    gold_q10.alias("s"),
    """
    t.machineID = s.machineID
    AND t.component = s.component
    AND t.maintenance_datetime = s.maintenance_datetime
    """
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Q10 table updated using MERGE")

spark.table(
    "workspace.default.gold_q10_mtbf"
).printSchema()

print(
    "Gold Q10 row count:",
    spark.table(
        "workspace.default.gold_q10_mtbf"
    ).count()
)

Gold Q10 table updated using MERGE
root
 |-- machineID: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- component: string (nullable = true)
 |-- maintenance_datetime: timestamp (nullable = true)
 |-- next_failure_datetime: timestamp (nullable = true)
 |-- mtbf_days: integer (nullable = true)

Gold Q10 row count: 3286


In [0]:
display(
    model_component_mtbf
)

model,component,maintenance_events,average_mtbf_days
model1,comp1,59,123.13559322033899
model1,comp2,94,108.32978723404256
model1,comp3,94,88.92553191489361
model1,comp4,67,97.77611940298507
model2,comp1,58,107.25862068965517
model2,comp2,92,116.44565217391305
model2,comp3,89,109.82022471910112
model2,comp4,48,105.58333333333333
model3,comp1,143,139.04895104895104
model3,comp2,209,108.10047846889952


Databricks visualization. Run in Databricks to view.